# Isotope Peak Detection in NIST20 and NIST23

Validates `detect_isotope_peaks` from `fragnnet.utils.spec_utils` on both reference datasets.

**Questions answered:**
- What fraction of spectra contain isotope peaks?
- What fraction of all peaks are isotope peaks?
- How does this break down by instrument type, fragmentation mode, and precursor type?
- Do NIST20 and NIST23 differ in isotope prevalence?

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
from fragnnet.utils.spec_utils import detect_isotope_peaks

PROC_DP = Path("../data/proc")
SAMPLE_N = 100_000   # spectra per dataset; set to None to use all
RANDOM_SEED = 42

## Helper — per-spectrum isotope stats

In [ ]:
def compute_isotope_stats(peaks, mz_tol: float = 0.02, max_isotope: int = 3) -> dict:
    """Return isotope detection stats for a single spectrum's peaks list."""
    if not peaks:
        return {"n_peaks": 0, "n_isotope": 0, "frac_isotope": np.nan, "has_isotope": False}
    mzs = np.array([p[0] for p in peaks], dtype=np.float64)
    ints = np.array([p[1] for p in peaks], dtype=np.float64)
    mask = detect_isotope_peaks(mzs, ints, mz_tol=mz_tol, max_isotope=max_isotope)
    n_iso = int(mask.sum())
    n_total = len(mzs)
    return {
        "n_peaks": n_total,
        "n_isotope": n_iso,
        "frac_isotope": n_iso / n_total if n_total > 0 else np.nan,
        "has_isotope": n_iso > 0,
    }


def run_isotope_analysis(spec_df: pd.DataFrame, sample_n: int | None, seed: int = 42) -> pd.DataFrame:
    """Load, optionally sample, and compute isotope stats for a spec_df."""
    if sample_n is not None and len(spec_df) > sample_n:
        spec_df = spec_df.sample(sample_n, random_state=seed).reset_index(drop=True)
        print(f"  Sampled {sample_n:,} spectra")
    else:
        print(f"  Using all {len(spec_df):,} spectra")

    print("  Computing isotope stats…")
    stats = spec_df["peaks"].apply(compute_isotope_stats).apply(pd.Series)
    meta_cols = [c for c in ["spec_id", "mol_id", "inst_type", "frag_mode", "prec_type",
                              "ion_mode", "ace", "nce"] if c in spec_df.columns]
    result = pd.concat([spec_df[meta_cols].reset_index(drop=True), stats], axis=1)
    print(f"  Done. {result['has_isotope'].sum():,} / {len(result):,} spectra have ≥1 isotope peak "
          f"({100 * result['has_isotope'].mean():.1f}%)")
    return result

## NIST20

In [ ]:
print("Loading NIST20 spec_df…")
spec20 = pd.read_pickle(PROC_DP / "nist20/spec_df.pkl")[
    ["spec_id", "mol_id", "inst_type", "frag_mode", "prec_type", "ion_mode", "ace", "nce", "peaks"]
].copy()
print(f"  Total spectra: {len(spec20):,}")
print("  inst_type counts:")
print(spec20["inst_type"].value_counts().to_string())

stats20 = run_isotope_analysis(spec20, SAMPLE_N, RANDOM_SEED)

## NIST23

In [ ]:
print("Loading NIST23 spec_df…")
spec23 = pd.read_pickle(PROC_DP / "nist23/spec_df.pkl")[
    ["spec_id", "mol_id", "inst_type", "frag_mode", "prec_type", "ion_mode", "ace", "nce", "peaks"]
].copy()
print(f"  Total spectra: {len(spec23):,}")
print("  inst_type counts:")
print(spec23["inst_type"].value_counts().to_string())

stats23 = run_isotope_analysis(spec23, SAMPLE_N, RANDOM_SEED)

## Summary statistics

In [ ]:
def summary_table(stats: pd.DataFrame, label: str) -> pd.DataFrame:
    rows = []
    rows.append({
        "dataset": label, "group": "ALL", "value": "all",
        "n_spectra": len(stats),
        "pct_has_isotope": 100 * stats["has_isotope"].mean(),
        "mean_frac_isotope": stats["frac_isotope"].mean(),
        "median_n_isotope": stats["n_isotope"].median(),
    })
    for group_col in ["inst_type", "frag_mode", "prec_type"]:
        if group_col not in stats.columns:
            continue
        for val, sub in stats.groupby(group_col, dropna=True):
            rows.append({
                "dataset": label, "group": group_col, "value": val,
                "n_spectra": len(sub),
                "pct_has_isotope": 100 * sub["has_isotope"].mean(),
                "mean_frac_isotope": sub["frac_isotope"].mean(),
                "median_n_isotope": sub["n_isotope"].median(),
            })
    return pd.DataFrame(rows)


tbl20 = summary_table(stats20, "NIST20")
tbl23 = summary_table(stats23, "NIST23")
tbl_all = pd.concat([tbl20, tbl23], ignore_index=True)

with pd.option_context("display.float_format", "{:.3f}".format, "display.max_rows", 60):
    display(tbl_all)

## Plots

In [ ]:
# ── Overview: % spectra with isotope peaks by inst_type ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)

for ax, (stats, label) in zip(axes, [(stats20, "NIST20"), (stats23, "NIST23")]):
    grp = (
        stats.groupby("inst_type", dropna=True)["has_isotope"]
        .agg(["mean", "count"])
        .sort_values("mean", ascending=False)
        .reset_index()
    )
    grp["pct"] = grp["mean"] * 100
    bars = ax.bar(range(len(grp)), grp["pct"], color="steelblue", alpha=0.8)
    for i, row in grp.iterrows():
        ax.text(i, row["pct"] + 0.3, f"n={row['count']:,}", ha="center", fontsize=7)
    ax.set_xticks(range(len(grp)))
    ax.set_xticklabels(grp["inst_type"], rotation=20, ha="right", fontsize=9)
    ax.set_ylabel("% spectra with ≥1 isotope peak")
    ax.set_title(f"{label} — by instrument type")
    ax.set_ylim(0, max(grp["pct"].max() * 1.15, 5))

plt.suptitle("Fraction of spectra with detected isotope peaks", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── By frag_mode ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)

for ax, (stats, label) in zip(axes, [(stats20, "NIST20"), (stats23, "NIST23")]):
    if "frag_mode" not in stats.columns:
        ax.set_title(f"{label} — frag_mode not available")
        continue
    grp = (
        stats.groupby("frag_mode", dropna=True)["has_isotope"]
        .agg(["mean", "count"])
        .sort_values("mean", ascending=False)
        .reset_index()
    )
    grp["pct"] = grp["mean"] * 100
    ax.bar(range(len(grp)), grp["pct"], color="darkorange", alpha=0.8)
    for i, row in grp.iterrows():
        ax.text(i, row["pct"] + 0.3, f"n={row['count']:,}", ha="center", fontsize=7)
    ax.set_xticks(range(len(grp)))
    ax.set_xticklabels(grp["frag_mode"], rotation=20, ha="right", fontsize=9)
    ax.set_ylabel("% spectra with ≥1 isotope peak")
    ax.set_title(f"{label} — by fragmentation mode")
    ax.set_ylim(0, max(grp["pct"].max() * 1.15, 5))

plt.suptitle("Isotope peak prevalence by fragmentation mode", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── By prec_type ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)

for ax, (stats, label) in zip(axes, [(stats20, "NIST20"), (stats23, "NIST23")]):
    grp = (
        stats.groupby("prec_type", dropna=True)["has_isotope"]
        .agg(["mean", "count"])
        .sort_values("mean", ascending=False)
        .reset_index()
    )
    grp["pct"] = grp["mean"] * 100
    ax.bar(range(len(grp)), grp["pct"], color="mediumseagreen", alpha=0.8)
    for i, row in grp.iterrows():
        ax.text(i, row["pct"] + 0.3, f"n={row['count']:,}", ha="center", fontsize=7)
    ax.set_xticks(range(len(grp)))
    ax.set_xticklabels(grp["prec_type"], rotation=20, ha="right", fontsize=9)
    ax.set_ylabel("% spectra with ≥1 isotope peak")
    ax.set_title(f"{label} — by precursor type")
    ax.set_ylim(0, max(grp["pct"].max() * 1.15, 5))

plt.suptitle("Isotope peak prevalence by precursor type", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Distribution: number of isotope peaks per spectrum ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (stats, label) in zip(axes, [(stats20, "NIST20"), (stats23, "NIST23")]):
    counts = stats["n_isotope"].value_counts().sort_index()
    # Show 0..10, group rest
    MAX_SHOW = 10
    low = counts[counts.index <= MAX_SHOW]
    high_sum = counts[counts.index > MAX_SHOW].sum()
    xs = list(low.index) + ([f">{MAX_SHOW}"] if high_sum > 0 else [])
    ys = list(low.values) + ([high_sum] if high_sum > 0 else [])
    ax.bar(range(len(xs)), ys, color="slateblue", alpha=0.8)
    ax.set_xticks(range(len(xs)))
    ax.set_xticklabels(xs, fontsize=9)
    ax.set_xlabel("Number of isotope peaks per spectrum")
    ax.set_ylabel("Count of spectra")
    ax.set_title(f"{label}")
    ax.set_yscale("log")

plt.suptitle("Distribution of isotope peak count per spectrum (log scale)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Distribution: fraction of peaks that are isotope peaks ───────────────────
# Only for spectra that have ≥1 isotope peak
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (stats, label) in zip(axes, [(stats20, "NIST20"), (stats23, "NIST23")]):
    sub = stats[stats["has_isotope"] & stats["frac_isotope"].notna()]["frac_isotope"]
    ax.hist(sub, bins=40, range=(0, 1), color="tomato", alpha=0.7, edgecolor="white")
    ax.axvline(sub.median(), color="black", lw=1.5, ls="--", label=f"median={sub.median():.3f}")
    ax.axvline(sub.mean(), color="navy", lw=1.5, ls="-.", label=f"mean={sub.mean():.3f}")
    ax.set_xlabel("Fraction of peaks that are isotope peaks")
    ax.set_ylabel("Count of spectra")
    ax.set_title(f"{label} (n={len(sub):,} spectra with isotopes)")
    ax.legend(fontsize=9)

plt.suptitle("Isotope fraction distribution (spectra with ≥1 isotope peak)", y=1.02)
plt.tight_layout()
plt.show()

## Example spectra with isotope peaks highlighted

In [ ]:
def plot_spectrum_with_isotopes(
    ax, peaks, title: str = "", mz_tol: float = 0.02, max_isotope: int = 3
) -> None:
    """Plot a spectrum with isotope peaks highlighted in red."""
    mzs = np.array([p[0] for p in peaks], dtype=np.float64)
    ints = np.array([p[1] for p in peaks], dtype=np.float64)
    ints_norm = ints / ints.max() * 100
    mask = detect_isotope_peaks(mzs, ints, mz_tol=mz_tol, max_isotope=max_isotope)

    for mz, intensity, is_iso in zip(mzs, ints_norm, mask):
        color = "red" if is_iso else "steelblue"
        ax.vlines(mz, 0, intensity, color=color, lw=1.2, alpha=0.85)

    n_iso = mask.sum()
    ax.set_xlabel("m/z")
    ax.set_ylabel("Relative intensity (%)")
    ax.set_title(f"{title}  [{n_iso}/{len(mzs)} isotope peaks]", fontsize=9)
    # Legend proxies
    from matplotlib.lines import Line2D
    ax.legend(
        handles=[
            Line2D([0], [0], color="steelblue", lw=2, label="fragment"),
            Line2D([0], [0], color="red", lw=2, label="isotope"),
        ],
        fontsize=7,
    )


# Pull 6 spectra that have isotope peaks (from NIST23)
example_ids = stats23[stats23["n_isotope"] >= 2]["spec_id"].sample(6, random_state=0).values
ex_peaks = (
    spec23[spec23["spec_id"].isin(example_ids)][["spec_id", "peaks", "inst_type", "prec_type"]]
    .set_index("spec_id")
    .loc[example_ids]
)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, (spec_id, row) in zip(axes.flat, ex_peaks.iterrows()):
    title = f"NIST23 spec {spec_id} | {row['inst_type']} | {row['prec_type']}"
    plot_spectrum_with_isotopes(ax, row["peaks"], title=title)
plt.suptitle("Example spectra: fragment peaks (blue) vs isotope peaks (red)", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Same for NIST20
example_ids20 = stats20[stats20["n_isotope"] >= 2]["spec_id"].sample(6, random_state=1).values
ex_peaks20 = (
    spec20[spec20["spec_id"].isin(example_ids20)][["spec_id", "peaks", "inst_type", "prec_type"]]
    .set_index("spec_id")
    .loc[example_ids20]
)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, (spec_id, row) in zip(axes.flat, ex_peaks20.iterrows()):
    title = f"NIST20 spec {spec_id} | {row['inst_type']} | {row['prec_type']}"
    plot_spectrum_with_isotopes(ax, row["peaks"], title=title)
plt.suptitle("NIST20 example spectra: fragment peaks (blue) vs isotope peaks (red)", y=1.01)
plt.tight_layout()
plt.show()

## NIST20 vs NIST23 comparison

In [ ]:
# Side-by-side comparison by inst_type (only types present in both datasets)
def pct_by_group(stats, col):
    return (
        stats.groupby(col, dropna=True)["has_isotope"]
        .mean()
        .mul(100)
        .rename("pct")
        .reset_index()
    )


g20 = pct_by_group(stats20, "inst_type").rename(columns={"pct": "NIST20"})
g23 = pct_by_group(stats23, "inst_type").rename(columns={"pct": "NIST23"})
comp = g20.merge(g23, on="inst_type", how="outer").set_index("inst_type").fillna(0)

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(comp))
w = 0.35
ax.bar(x - w / 2, comp["NIST20"], w, label="NIST20", color="steelblue", alpha=0.85)
ax.bar(x + w / 2, comp["NIST23"], w, label="NIST23", color="darkorange", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(comp.index, rotation=20, ha="right")
ax.set_ylabel("% spectra with ≥1 isotope peak")
ax.set_title("Isotope prevalence: NIST20 vs NIST23 by instrument type")
ax.legend()
plt.tight_layout()
plt.show()

# Print numeric table
print(comp.round(2).to_string())

## Sensitivity to `mz_tol` and `check_intensity`

In [ ]:
# How sensitive are results to the tolerance and intensity-check flag?
# Use a small NIST23 sub-sample for speed.
SENS_N = 10_000
sens_sample = spec23.sample(SENS_N, random_state=0)

configs = [
    {"mz_tol": 0.01, "check_intensity": True},
    {"mz_tol": 0.02, "check_intensity": True},   # default
    {"mz_tol": 0.05, "check_intensity": True},
    {"mz_tol": 0.02, "check_intensity": False},
]

rows = []
for cfg in configs:
    pct = sens_sample["peaks"].apply(
        lambda p: compute_isotope_stats(p, **cfg)["has_isotope"]
    ).mean() * 100
    mean_n = sens_sample["peaks"].apply(
        lambda p: compute_isotope_stats(p, **cfg)["n_isotope"]
    ).mean()
    rows.append({**cfg, "pct_has_isotope": pct, "mean_n_isotope": mean_n})

sens_df = pd.DataFrame(rows)
print("Sensitivity analysis (NIST23 sample, n={:,}):".format(SENS_N))
display(sens_df.round(3))